# Lεtree training

Trains the correction model on a Colab GPU. Datasets and runs are kept in Google
Drive, so a dataset is generated once and an interrupted run resumes.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

!git clone -q https://github.com/KadirKess/letree.git /content/letree || git -C /content/letree pull -q
%cd /content/letree
!pip install -q --no-deps -e .
!pip install -q torch-geometric higra
!git describe --always --dirty

In [ ]:
!pip install -q pytest
!python -m pytest -q

In [ ]:
DRIVE = "/content/drive/MyDrive/letree"
DATASET = "squares_gaussian_32_std8_zone"
SQUARES = "--height 32 --width 32 --min-squares 16 --max-squares 48 --max-side 8"
NOISES = "additive_gaussian --gaussian-std 8"
NOISY_PER_TRAIN_IMAGE = 2
MODEL = "--width 64 --precision fp32 --aggregations mean max min"
LOSSES = "--zone-pointer --separating-weight 5"
CLEAN_ONLY = False
MAX_EPOCHS = 300
STAGE = "joint"
OBSERVED_RUN = "32_gaussian_std8_2_observed"
RUN = "32_gaussian_std8_2_zone_pointer_w5"

In [ ]:
import os

archive = f"{DRIVE}/datasets/{DATASET}.tar"
dataset = f"/content/datasets/{DATASET}"
os.makedirs(f"{DRIVE}/datasets", exist_ok=True)

if os.path.exists(dataset):
    print("Dataset already extracted")
elif os.path.exists(archive):
    !mkdir -p /content/datasets && tar -xf {archive} -C /content/datasets
else:
    !python scripts/make_squares.py /content/sources/squares {SQUARES}
    !python scripts/make_dataset.py /content/sources/squares {dataset} --noise {NOISES} --noisy {NOISY_PER_TRAIN_IMAGE}
    !tar -cf {archive} -C /content/datasets {DATASET}

In [ ]:
run = f"{DRIVE}/runs/{RUN}"
flags = "--clean-only" if CLEAN_ONLY else ""
flags += f" --stage {STAGE}"
if STAGE == "frozen":
    flags += f" --observed-run {DRIVE}/runs/{OBSERVED_RUN}"
if os.path.exists(f"{run}/last.pt"):
    flags += " --resume"

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/train.py {dataset} {run} --max-epochs {MAX_EPOCHS} --attention flex {MODEL} --compile model --compile-mode max-autotune-no-cudagraphs --workers 4 {LOSSES} {flags} --plateau-patience 10 --early-stopping-patience 20

In [ ]:
import json

import pandas as pd
import torch

history = json.load(open(f"{run}/history.json"))
best = torch.load(f"{run}/best.pt", map_location="cpu")["history"][-1]
validation = best["validation"]
quantities = [
    "observed_parent_accuracy",
    "set_pointer_accuracy",
    "increment_accuracy",
    "joint_accuracy",
    "edge_accuracy",
]
quantities = [name for name in quantities if f"all/{name}/image_mean" in validation]
groups = [
    name.removesuffix("/images") for name in validation if name.endswith("/images")
]

print(f"{len(history)} epochs, best validation at epoch {best['epoch']}")
print(f"selection (validation loss) {validation['selection']:.4f}")
pd.DataFrame(
    {
        (group, weighting): [
            validation[f"{group}/{name}/{weighting}"] for name in quantities
        ]
        for group in groups
        for weighting in ("image_mean", "pooled")
    },
    index=quantities,
).round(3)

## Test evaluation

Decodes the best checkpoint's predictions on the test split with the soft and the hard
vote and scores them, with the observed tree and the oracle targets as references.
Records and the summary are written into the run folder.

In [ ]:
run = f"{DRIVE}/runs/{RUN}"
!python scripts/evaluate.py {run} {dataset} --split test --workers 4

In [ ]:
import json

import pandas as pd

from letree.evaluation.records import SOURCES

summary = json.load(open(f"{run}/test_best_summary.json"))
groups = list(
    dict.fromkeys(
        key.split("/")[1] for key in summary if key.endswith("/every/samples")
    )
)
columns = {
    "coverage": "coverage",
    "exact": "exact_recovery",
    "roots": "root_claims/sample_mean",
    "groups": "groups/sample_mean",
    "lost zones": "lost_clean_zones/sample_mean",
    "misassigned": "all/misassigned_pixels/sample_mean",
    "depth MAE": "all/depth_mae/sample_mean",
    "exact depth": "all/exact_depth/sample_mean",
    "root lost": "clean_root_lost",
}
rows = {
    (source, group): {
        label: summary.get(f"{source}/{group}/every/{name}")
        for label, name in columns.items()
    }
    for group in groups
    for source in SOURCES
    if f"{source}/{group}/every/samples" in summary
}
pd.DataFrame(rows).T.round(4)

In [ ]:
!python scripts/plot_run.py {run} {dataset} --per-group 2